In [15]:
%pip install geopandas folium

  Using cached folium-0.20.0-py2.py3-none-any.whl.metadata (4.2 kB)
  Using cached branca-0.8.2-py3-none-any.whl.metadata (1.7 kB)
Using cached folium-0.20.0-py2.py3-none-any.whl (113 kB)
Using cached branca-0.8.2-py3-none-any.whl (26 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.7/24.7 MB 317.3 kB/s  0:01:01m0:00:0100:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 328.8 kB/s  0:00:16 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 162.7 kB/s  0:00:08m0:00:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [folium]2m5/6 [folium]]
Note: you may need to restart the kernel to use updated packages.


In [29]:
from pathlib import Path

print("Current working directory:", Path.cwd())

# Find the GeoJSON file
files = list(Path.home().rglob("LowWaterCrossings*.geojson"))

for file in files:
    print(file)
    

Current working directory: /Users/minhanhdoan/FloodGuard---Datathon26/notebooks
/Users/minhanhdoan/FloodGuard---Datathon26/data/LowWaterCrossings_-2415515138282452691.geojson
/Users/minhanhdoan/FloodGuard---Datathon26/data/.ipynb_checkpoints/LowWaterCrossings_-2415515138282452691-checkpoint.geojson


In [37]:
import geopandas as gpd
from pathlib import Path

# Go up one folder, then into data/
file_path = Path("../data/LowWaterCrossings_-2415515138282452691.geojson")

# Read the GeoJSON file
gdf = gpd.read_file(file_path)

print("Number of crossings:", len(gdf))
print("Coordinate system:", gdf.crs)# Keep the columns needed for our dashboard
crossings = gdf[
    ["OBJECTID", "Name", "CartID", "geometry"]
].copy()

# Remove records without valid geometry
crossings = crossings.dropna(subset=["geometry"])
crossings = crossings[~crossings.geometry.is_empty].copy()

# Check duplicate IDs
print("Duplicate IDs:", crossings["CartID"].duplicated().sum())

# Rename columns for our backend
crossings = crossings.rename(columns={
    "CartID": "location_id",
    "Name": "name"
})

# Extract latitude and longitude
crossings["latitude"] = crossings.geometry.y
crossings["longitude"] = crossings.geometry.x

display(crossings.head())

print("Cleaned crossings:", len(crossings))
print("Columns:", gdf.columns.tolist())

display(gdf.head())

Number of crossings: 156
Coordinate system: EPSG:4326
Duplicate IDs: 0


,OBJECTID,name,location_id,geometry,latitude,longitude
0,1,Elm Creek at Jones Maltsberger,LWC_7070122,POINT (-98.43465 29.59547),29.595473,-98.434648
1,2,Cougar Way Fort Sam Houston,LWC_7070275,POINT (-98.41729 29.47969),29.479689,-98.417294
2,3,Pinn Rd 1320 ft S of West Commerce,LWC_7070194,POINT (-98.6237 29.43281),29.432810,-98.623701
3,4,Dreamland at RR Crossing,LWC_7070146,POINT (-98.53996 29.53693),29.536935,-98.539958
4,5,Hausman Rd 4200 ft W of IH-10,LWC_7070105,POINT (-98.60747 29.57244),29.572436,-98.607470


Cleaned crossings: 156
Columns: ['OBJECTID', 'Name', 'CartID', 'GlobalID', 'geometry']


,OBJECTID,Name,CartID,GlobalID,geometry
0,1,Elm Creek at Jones Maltsberger,LWC_7070122,8f85f35d-04d2-4206-974c-2c94272e0ef9,POINT (-98.43465 29.59547)
1,2,Cougar Way Fort Sam Houston,LWC_7070275,89ccdeec-4490-47fb-b29d-f7a89549ad91,POINT (-98.41729 29.47969)
2,3,Pinn Rd 1320 ft S of West Commerce,LWC_7070194,39436028-e662-4065-9fe0-4ae520f53342,POINT (-98.6237 29.43281)
3,4,Dreamland at RR Crossing,LWC_7070146,d89d9137-1c52-4336-9681-713ca7c0bd76,POINT (-98.53996 29.53693)
4,5,Hausman Rd 4200 ft W of IH-10,LWC_7070105,79a06776-a1d3-46f9-bc68-783e60812d48,POINT (-98.60747 29.57244)


In [38]:
import folium
from folium.plugins import MarkerCluster

# Center map on San Antonio
m = folium.Map(
    location=[29.4241, -98.4936],
    zoom_start=11,
    tiles="OpenStreetMap"
)

# Cluster nearby crossings
marker_cluster = MarkerCluster().add_to(m)

for _, row in crossings.iterrows():
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=5,
        color="#2563eb",
        fill=True,
        fill_color="#3b82f6",
        fill_opacity=0.8,
        popup=folium.Popup(
            f"<b>{row['name']}</b><br>"
            f"ID: {row['location_id']}",
            max_width=300
        ),
        tooltip=row["name"]
    ).add_to(marker_cluster)

display(m)

In [39]:
from pathlib import Path

output_path = Path("../data/locations.csv")

crossings[
    ["location_id", "name", "latitude", "longitude"]
].to_csv(output_path, index=False)

print("Saved to:", output_path)
display(crossings[
    ["location_id", "name", "latitude", "longitude"]
].head())

Saved to: ../data/locations.csv


,location_id,name,latitude,longitude
0,LWC_7070122,Elm Creek at Jones Maltsberger,29.595473,-98.434648
1,LWC_7070275,Cougar Way Fort Sam Houston,29.479689,-98.417294
2,LWC_7070194,Pinn Rd 1320 ft S of West Commerce,29.432810,-98.623701
3,LWC_7070146,Dreamland at RR Crossing,29.536935,-98.539958
4,LWC_7070105,Hausman Rd 4200 ft W of IH-10,29.572436,-98.607470
